# Knowledge Extraction from Atomically Resolved Images

**IMC-21: Artificial Intelligence Methods for Microscopy Analysis and Knowledge Extraction**

By Rama Vasudevan, CNMS, ORNL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pycroscopy/IMC21-Workshop/blob/main/notebooks/13_00_Knowledge_Extraction_FeSeTe.ipynb)

---

## Introduction

This notebook recreates the central idea of the paper by Vleck et al. [1] in an abbreviated form. We will simulate an atomically resolved STM image of the chalcogen surface of $FeSe_xTe_{1-x}$, recover the Se/Te arrangement, and infer an effective interaction energy from local configuration statistics.

We will work through how to setup the model, what the image actually encodes and how to use the correct loss functions to actually determine the interaction energy

[1] Vlcek, Lukas, et al. "Knowledge extraction from atomically resolved images." ACS nano 11.10 (2017): 10313-10320

<div style="max-width: 80%; border-left: 6px solid #2e7d32; border-top: 1px solid #e0e0e0; border-right: 1px solid #e0e0e0; border-bottom: 1px solid #e0e0e0; border-radius: 4px; padding: 0; margin-bottom: 20px; box-shadow: 0 4px 8px rgba(0,0,0,0.1), 0 1px 3px rgba(0,0,0,0.08); background-color: #ffffff;">
  <div style="background-color: transparent; color: #1b5e20; padding: 10px 15px; font-weight: bold; border-bottom: 1px solid #e0e0e0; display: flex; align-items: center; gap: 8px;">
    <span>Target</span> Learning Goals
  </div>
  <div style="padding: 15px; background-color: transparent; color: #333333;">
  <ul style="margin: 0; padding-left: 20px;">
      <li style="margin-bottom: 8px;">Connect atomically resolved images to local structural descriptors.</li>
      <li style="margin-bottom: 8px;">Use `pycroscopy` windowing to turn an image into a matrix of local observations.</li>
      <li style="margin-bottom: 8px;">Compare a physics-defined codebook with a data-driven k-means codebook.</li>
      <li style="margin-bottom: 0;">Estimate how image noise propagates into uncertainty on interaction energies and compare different loss functions, and predict microstaets for different thermodynamic conditions.</li>
    </ul>
  </div>
</div>

## 0. Setup and imports. 

Here we will install and import the necessary packages.

In [ ]:
# --- Setup: Colab-safe. Skip if you already have these packages. ---
# This follows the same pycroscopy install pattern as the 9_00 intro notebook:
import io, contextlib, warnings, itertools, os, sys, subprocess

try:
    import pycroscopy, sidpy
except Exception as exc:
    print('Installing pycroscopy stack with the same no-deps pattern used in the intro notebook:', repr(exc))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sidpy', 'pysptools', 'tensorly'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'pycroscopy'], check=True)
    import pycroscopy, sidpy

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from pycroscopy.image import ImageWindowing

warnings.filterwarnings('ignore', module=r'sidpy.*')
plt.rcParams.update({'figure.dpi': 110, 'image.cmap': 'afmhot'})
rng = np.random.default_rng(7)
print('pycroscopy', pycroscopy.__version__, '| sidpy', sidpy.__version__)

## 1. Simulate the FeSeTe surface

$FeSe_xTe_{1-x}$ cleaves to expose a square chalcogen lattice. The paper studies $FeSe_{0.45}Te_{0.55}$ and finds clustering of like atoms. We will mimic this with a binary lattice model where unlike Se-Te nearest-neighbor pairs cost an energy $w_0$ in units of $k_B T$. The Monte Carlo moves swap Se and Te atoms, so the global composition stays fixed. Larger positive $w_0$ means stronger segregation. You can choose to play with $w_0$ or the composition if you so wish.

In [ ]:
def neighbor_sum_energy(spins, w0=0.5):
    """Energy penalty w0 for unlike horizontal/vertical nearest-neighbor pairs."""
    return w0 * ((spins != np.roll(spins, 1, axis=0)).sum() +
                 (spins != np.roll(spins, 1, axis=1)).sum())


def make_random_lattice(n=48, x_se=0.45, rng=None):
    """Binary lattice: 1 = Se (bright), 0 = Te (dark)."""
    rng = np.random.default_rng() if rng is None else rng
    flat = np.zeros(n*n, dtype=np.int8)
    flat[:int(round(x_se*n*n))] = 1
    rng.shuffle(flat)
    return flat.reshape(n, n)


def local_pair_energy(spins, r, c, w0):
    val = spins[r, c]
    nbrs = [spins[(r-1) % spins.shape[0], c], spins[(r+1) % spins.shape[0], c],
            spins[r, (c-1) % spins.shape[1]], spins[r, (c+1) % spins.shape[1]]]
    return w0 * sum(val != nb for nb in nbrs)


def kawasaki_mc(n=48, x_se=0.45, w0=0.55, sweeps=500, burn=100, sample_every=25, rng=None):
    """Fixed-composition Metropolis MC for a binary lattice."""
    rng = np.random.default_rng() if rng is None else rng
    spins = make_random_lattice(n, x_se, rng)
    samples = []
    n_moves = n*n
    for sweep in range(sweeps):
        for _ in range(n_moves):
            r1, c1 = rng.integers(0, n, 2)
            r2, c2 = rng.integers(0, n, 2)
            if spins[r1, c1] == spins[r2, c2]:
                continue
            before = local_pair_energy(spins, r1, c1, w0) + local_pair_energy(spins, r2, c2, w0)
            spins[r1, c1], spins[r2, c2] = spins[r2, c2], spins[r1, c1]
            after = local_pair_energy(spins, r1, c1, w0) + local_pair_energy(spins, r2, c2, w0)
            dE = after - before
            if dE > 0 and rng.random() > np.exp(-dE):
                spins[r1, c1], spins[r2, c2] = spins[r2, c2], spins[r1, c1]
        if sweep >= burn and (sweep - burn) % sample_every == 0:
            samples.append(spins.copy())
    return spins, samples

n_atoms = 48
x_se = 0.45
true_w0 = 0.55
surface, mc_samples = kawasaki_mc(n=n_atoms, x_se=x_se, w0=true_w0, sweeps=450, burn=150,
                                  sample_every=30, rng=rng)
print(f'{len(mc_samples)} MC samples collected; final Se fraction = {surface.mean():.3f}')
print(f'Final unlike-pair energy / site = {neighbor_sum_energy(surface, true_w0) / surface.size:.3f} kBT')

fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
ax[0].imshow(surface, cmap='coolwarm', vmin=0, vmax=1, interpolation='nearest')
ax[0].set_title('simulated chalcogen lattice\n1 = Se, 0 = Te')
ax[1].imshow(gaussian_filter(surface.astype(float), 1.2), cmap='coolwarm', vmin=0, vmax=1)
ax[1].set_title('same lattice, visually smoothed')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## 2. Render a simulated STM image

The real STM image in the paper contains bright and dark chalcogen sites. Here Se columns are brighter than Te columns, each site is rendered as a Gaussian peak, and the image includes background, blur, and Gaussian noise.

In [ ]:
def render_stm(spins, pitch=8, sigma=1.25, noise=0.12, background=0.12, rng=None):
    """Render a binary atom lattice as an STM-like image with known atom centers."""
    rng = np.random.default_rng() if rng is None else rng
    ny, nx = spins.shape
    h, w = ny*pitch, nx*pitch
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.zeros((h, w), dtype=float)
    centers = []
    for r in range(ny):
        for c in range(nx):
            y = r*pitch + pitch/2
            x = c*pitch + pitch/2
            amp = 1.25 if spins[r, c] else 0.70
            amp *= rng.normal(1.0, 0.04)
            img += amp * np.exp(-((xx-x)**2 + (yy-y)**2)/(2*sigma**2))
            centers.append((y, x))
    slow = background * (np.sin(2*np.pi*xx/w*1.1) + 0.7*np.cos(2*np.pi*yy/h*0.8))
    img = img + slow + rng.normal(0, noise, img.shape)
    img = img - img.min()
    img = img / img.max()
    return img, np.array(centers).reshape(ny, nx, 2)

pitch = 8
noise_sigma = 0.12
stm_image, centers = render_stm(surface, pitch=pitch, noise=noise_sigma, rng=rng)

fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
ax[0].imshow(stm_image, cmap='gray')
ax[0].set_title('simulated STM image')
ax[1].imshow(stm_image[:12*pitch, :12*pitch], cmap='gray')
ax[1].set_title('zoom: bright/dark atom sites')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

## 3. Physics 'codebook', i.e. the 12 nearest-neighbor configurations we use for comparisons

For every central atom, we examine its four nearest neighbors: north, east, south, and west. There are $2^5 = 32$ raw binary patterns, but rotations and mirror symmetries of the square lattice reduce them to 12 distinct local configurations. These are the bins of the "physics codebook" that we will use. As we change the interaction energies we can see the effect they produce on these histograms.

In [ ]:
def transforms_4(v):
    """All rotations/reflections of a four-neighbor ring ordered N,E,S,W."""
    v = tuple(v)
    rots = [v[i:] + v[:i] for i in range(4)]
    refl = tuple([v[0], v[3], v[2], v[1]])
    rots += [refl[i:] + refl[:i] for i in range(4)]
    return rots


def canonical_config(center, neighbors):
    return (int(center), min(transforms_4(tuple(int(x) for x in neighbors))))

# Enumerate the complete symmetry-reduced nearest-neighbor codebook.
physics_codebook = []
for center in [0, 1]:
    reps = sorted({canonical_config(center, nbs) for nbs in itertools.product([0, 1], repeat=4)})
    physics_codebook.extend(reps)
code_to_index = {cfg: i for i, cfg in enumerate(physics_codebook)}
print(f'{len(physics_codebook)} symmetry-distinct configurations')
for i, cfg in enumerate(physics_codebook):
    print(f'{i:02d}: center={cfg[0]}, neighbors(N,E,S,W)={cfg[1]}')


def local_config_indices(spins):
    idx = []
    for r in range(spins.shape[0]):
        for c in range(spins.shape[1]):
            nbs = (spins[(r-1) % spins.shape[0], c], spins[r, (c+1) % spins.shape[1]],
                   spins[(r+1) % spins.shape[0], c], spins[r, (c-1) % spins.shape[1]])
            idx.append(code_to_index[canonical_config(spins[r, c], nbs)])
    return np.array(idx, dtype=int)


def config_histogram(spins, normalize=True):
    counts = np.bincount(local_config_indices(spins), minlength=len(physics_codebook)).astype(float)
    if normalize:
        counts /= counts.sum()
    return counts


def plot_codebook(codebook):
    fig, ax = plt.subplots(2, 6, figsize=(10, 3.8))
    for i, (center, nbs) in enumerate(codebook):
        a = ax.ravel()[i]
        patch = np.full((3, 3), np.nan)
        patch[1, 1] = center
        patch[0, 1], patch[1, 2], patch[2, 1], patch[1, 0] = nbs
        a.imshow(patch, cmap='coolwarm', vmin=0, vmax=1)
        a.set_title(str(i), fontsize=10)
        a.set_xticks([]); a.set_yticks([])
    plt.suptitle('physics codebook: 12 symmetry-distinct nearest-neighbor configurations')
    plt.tight_layout()

plot_codebook(physics_codebook)

In [ ]:
physics_hist = config_histogram(surface)

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(np.arange(len(physics_hist)), physics_hist, color='steelblue')
ax.set_xlabel('configuration index')
ax.set_ylabel('relative frequency')
ax.set_title('target histogram from the known simulated lattice')
ax.set_xticks(np.arange(len(physics_hist)))
plt.tight_layout()

## 4. Statistical distance fit of the interaction energy

The paper compares experimental and simulated histograms using statistical distance,

$$s(p,q) = \arccos \left( \sum_i \sqrt{p_i q_i} \right),$$

where $p_i$ and $q_i$ are probabilities for local configuration bin $i$. This is closely related to the Bhattacharyya coefficient and treats the two histograms symmetrically.

Rather than calculate the entire loss landscape, for the workshop, we will build a small model library by simulating several trial values of $w_0$ and choose the one with minimum distance to the target histogram. In reality you will want to use either the perturbation method noted in the paper or some other optimization method to obtain a better estimate. 

In [ ]:
def statistical_distance(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / p.sum()
    q = q / q.sum()
    bc = np.sqrt(p*q).sum()
    return np.arccos(np.clip(bc, 0, 1))


def simulate_histogram_at_w(w0, n=48, x_se=0.45, sweeps=260, burn=100, sample_every=40, seed=0):
    rr = np.random.default_rng(seed)
    final, samples = kawasaki_mc(n=n, x_se=x_se, w0=w0, sweeps=sweeps, burn=burn,
                                 sample_every=sample_every, rng=rr)
    hists = [config_histogram(s) for s in samples]
    if not hists:
        hists = [config_histogram(final)]
    return np.mean(hists, axis=0)

w_grid = np.linspace(0.0, 1.2, 17)
model_hists = []
for i, w in enumerate(w_grid):
    model_hists.append(simulate_histogram_at_w(w, n=n_atoms, x_se=x_se, seed=100+i))
model_hists = np.array(model_hists)

distances = np.array([statistical_distance(physics_hist, mh) for mh in model_hists])
best = distances.argmin()
est_w0 = w_grid[best]
print(f'true w0 = {true_w0:.2f} kBT')
print(f'estimated w0 from physics codebook = {est_w0:.2f} kBT')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(w_grid, distances**2, 'o-')
ax[0].axvline(true_w0, color='k', ls='--', label='true')
ax[0].axvline(est_w0, color='tab:red', ls=':', label='fit')
ax[0].set_xlabel('$w_0$ ($k_B T$)')
ax[0].set_ylabel('$s^2$')
ax[0].set_title('statistical-distance landscape')
ax[0].legend()

ax[1].bar(np.arange(12)-0.18, physics_hist, width=0.36, label='target')
ax[1].bar(np.arange(12)+0.18, model_hists[best], width=0.36, label='best model')
ax[1].set_xlabel('configuration index')
ax[1].set_ylabel('relative frequency')
ax[1].set_title('histogram match')
ax[1].legend()
plt.tight_layout()

## 5. Using windows to construct a codebook

The physics method assumes we already know the configurations to target. But this is not always the case - perhaps we dont know whether we should be looking only at nearest neighbors? Maybe we are unsure about symmetries? Or maybe we want to automate the entire process? As an alternative, here we use `pycroscopy.image.ImageWindowing` to extract one local image window per lattice site, then run k-means on the window intensities. As we saw in the early examples today, this will give us the characteristic local structures present. 

In [ ]:
def extract_neighborhood_windows(image, pitch=8, neighborhood_atoms=3):
    """Use pycroscopy ImageWindowing to get one 3x3-neighborhood window per interior atom."""
    ds = sidpy.Dataset.from_array(image, name='FeSeTe_STM')
    ds.data_type = 'image'
    window = neighborhood_atoms * pitch
    parms = dict(window_size_x=window, window_size_y=window,
                 window_step_x=pitch, window_step_y=pitch,
                 mode='image')
    with contextlib.redirect_stdout(io.StringIO()):
        windows = np.array(ImageWindowing(parms).MakeWindows(ds))
    return windows


def windows_to_matrix(windows):
    X = windows.reshape(windows.shape[0]*windows.shape[1], -1).astype(float)
    # Remove patch brightness offsets so k-means focuses more on local arrangement than background.
    X = X - X.mean(axis=1, keepdims=True)
    X = X / (X.std(axis=1, keepdims=True) + 1e-9)
    return X


def histogram_from_labels(labels, k):
    hist = np.bincount(labels, minlength=k).astype(float)
    return hist / hist.sum()


def statistical_distance(p, q):
    p = np.asarray(p, dtype=float) / np.sum(p)
    q = np.asarray(q, dtype=float) / np.sum(q)
    return np.arccos(np.clip(np.sqrt(p*q).sum(), 0, 1))

def render_codebook_centers(kmeans, scaler, window_shape, n_cols=6):
    centers = scaler.inverse_transform(kmeans.cluster_centers_)
    fig, ax = plt.subplots(int(np.ceil(len(centers)/n_cols)), n_cols, figsize=(10, 4.2))
    ax = np.asarray(ax).ravel()
    for i, center in enumerate(centers):
        im = center.reshape(window_shape)
        ax[i].imshow(im, cmap='gray')
        ax[i].set_title(f'codeword {i}')
        ax[i].set_xticks([]); ax[i].set_yticks([])
    for j in range(i+1, len(ax)):
        ax[j].axis('off')
    plt.suptitle('learned k-means window codebook')
    plt.tight_layout()

def learn_window_codebook(image, pitch=8, n_codewords=12, random_state=0):
    windows = extract_neighborhood_windows(image, pitch=pitch, neighborhood_atoms=3)
    X = windows_to_matrix(windows)
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    kmeans = KMeans(n_clusters=n_codewords, n_init=30, random_state=random_state).fit(Xs)
    hist = histogram_from_labels(kmeans.labels_, n_codewords)
    return {'windows': windows, 'X': X, 'scaler': scaler, 'kmeans': kmeans,
            'labels': kmeans.labels_, 'hist': hist}

n_codewords = 12
target_codebook = learn_window_codebook(stm_image, pitch=pitch, n_codewords=n_codewords)
print('Window tensor:', target_codebook['windows'].shape)
print('Target visual-codebook histogram:', np.round(target_codebook['hist'], 3))
render_codebook_centers(target_codebook['kmeans'], target_codebook['scaler'], target_codebook['windows'].shape[-2:])

## Compare models using the learned codebook

The codebook is learned once from the target image. For each simulated model at trial $w_0$, we render a synthetic STM image, extract windows the same way, assign each window to the nearest learned centroid, and compare the resulting histograms.

This is the direct analog of the physics-codebook comparison, except the bins are learned visual motifs.

In [ ]:
def assign_image_to_codebook(image, codebook, pitch=8):
    windows = extract_neighborhood_windows(image, pitch=pitch, neighborhood_atoms=3)
    X = windows_to_matrix(windows)
    Xs = codebook['scaler'].transform(X)
    labels = codebook['kmeans'].predict(Xs)
    hist = histogram_from_labels(labels, codebook['kmeans'].n_clusters)
    return hist, labels, windows


def simulate_visual_histogram_at_w(w0, codebook, n=48, x_se=0.45, pitch=8, noise=0.12,
                                   sweeps=260, burn=100, sample_every=40, seed=0):
    rr = np.random.default_rng(seed)
    final, samples = kawasaki_mc(n=n, x_se=x_se, w0=w0, sweeps=sweeps, burn=burn,
                                 sample_every=sample_every, rng=rr)
    hists = []
    for sample in samples if samples else [final]:
        img, _ = render_stm(sample, pitch=pitch, noise=noise, rng=rr)
        hist, _, _ = assign_image_to_codebook(img, codebook, pitch=pitch)
        hists.append(hist)
    return np.mean(hists, axis=0)

w_grid = np.linspace(0.0, 1.2, 13)
visual_model_hists = []
for i, w in enumerate(w_grid):
    visual_model_hists.append(simulate_visual_histogram_at_w(w, target_codebook, n=n_atoms, x_se=x_se,
                                                             pitch=pitch, noise=noise_sigma, seed=200+i))
visual_model_hists = np.array(visual_model_hists)
visual_distances = np.array([statistical_distance(target_codebook['hist'], h) for h in visual_model_hists])
best_idx = int(np.argmin(visual_distances))
visual_est_w0 = float(w_grid[best_idx])
print(f'true w0 = {true_w0:.2f} kBT')
print(f'estimated w0 from learned visual-window codebook = {visual_est_w0:.2f} kBT')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(w_grid, visual_distances**2, 'o-')
ax[0].axvline(true_w0, color='k', ls='--', label='true')
ax[0].axvline(visual_est_w0, color='tab:red', ls=':', label='fit')
ax[0].set_xlabel('$w_0$ ($k_B T$)')
ax[0].set_ylabel('$s^2$')
ax[0].set_title('visual-codebook distance landscape')
ax[0].legend()

x = np.arange(n_codewords)
ax[1].bar(x-0.18, target_codebook['hist'], width=0.36, label='target image')
ax[1].bar(x+0.18, visual_model_hists[best_idx], width=0.36, label='best model')
ax[1].set_xlabel('learned codeword index')
ax[1].set_ylabel('relative frequency')
ax[1].set_title('histogram match in learned codebook')
ax[1].legend()
plt.tight_layout()

## 6. Uncertainty from finite image area

The paper estimated error bars by dividing the image into 9 blocks. We can do the same. Here each block is assigned to the same learned visual-window codebook, giving a smaller histogram and a spread of fitted interaction energies.

In [ ]:
def block_visual_hists(image, codebook, n_blocks=3, pitch=8):
    hists = []
    nr, nc = image.shape
    br, bc = nr // n_blocks, nc // n_blocks
    for i in range(n_blocks):
        for j in range(n_blocks):
            block = image[i*br:(i+1)*br, j*bc:(j+1)*bc]
            hist, _, _ = assign_image_to_codebook(block, codebook, pitch=pitch)
            hists.append(hist)
    return np.array(hists)


def fit_w_from_hist(hist, model_hists=visual_model_hists, w_grid=w_grid):
    ds = np.array([statistical_distance(hist, mh) for mh in model_hists])
    return w_grid[ds.argmin()], ds.min()

block_estimates = np.array([fit_w_from_hist(h)[0] for h in block_visual_hists(stm_image, target_codebook, 3, pitch=pitch)])
print('block-wise w0 estimates:', np.round(block_estimates, 2))
print(f'mean +/- std = {block_estimates.mean():.2f} +/- {block_estimates.std(ddof=1):.2f} kBT')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(block_estimates, bins=np.arange(-0.03, 1.26, 0.12), color='tab:purple', alpha=0.8)
ax.axvline(true_w0, color='k', ls='--', label='true')
ax.set_xlabel('$w_0$ estimate ($k_B T$)')
ax.set_ylabel('number of image blocks')
ax.set_title('finite-area uncertainty')
ax.legend()
plt.tight_layout()

## 7. What happens when the image gets noisier?

Now we repeat the render -> ImageWindowing -> k-means labels -> histogram -> fit pipeline for several image noise levels. The physical lattice is unchanged, so any broadening of $w_0$ estimates is caused by difficulty in image classification

In [ ]:
def estimate_from_noisy_image(spins, noise, repeats=6, pitch=8, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    estimates, distances_to_target = [], []
    for rep in range(repeats):
        img, _ = render_stm(spins, pitch=pitch, noise=noise, rng=rng)
        noisy_codebook = learn_window_codebook(img, pitch=pitch, n_codewords=n_codewords, random_state=rep)
        noisy_model_hists = []
        for i, w in enumerate(w_grid):
            noisy_model_hists.append(simulate_visual_histogram_at_w(w, noisy_codebook, n=n_atoms, x_se=x_se,
                                                                     pitch=pitch, noise=noise, seed=1000+100*rep+i))
        noisy_model_hists = np.array(noisy_model_hists)
        noisy_distances = np.array([statistical_distance(noisy_codebook['hist'], h) for h in noisy_model_hists])
        best_idx = int(np.argmin(noisy_distances))
        estimates.append(float(w_grid[best_idx]))
        distances_to_target.append(float(statistical_distance(target_codebook['hist'], noisy_codebook['hist'])))
    return np.array(estimates), np.array(distances_to_target)

noise_levels = [0.05, 0.12, 0.20, 0.30]
noise_results = {}
for ns in noise_levels:
    est, codebook_drift = estimate_from_noisy_image(surface, ns, repeats=6, pitch=pitch, rng=rng)
    noise_results[ns] = (est, codebook_drift)
    print(f'noise={ns:.2f}: w0={est.mean():.2f} +/- {est.std(ddof=1):.2f} kBT, codebook drift={codebook_drift.mean():.3f}')

means = np.array([noise_results[ns][0].mean() for ns in noise_levels])
stds = np.array([noise_results[ns][0].std(ddof=1) for ns in noise_levels])
drifts = np.array([noise_results[ns][1].mean() for ns in noise_levels])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].errorbar(noise_levels, means, yerr=stds, marker='o', capsize=4)
ax[0].axhline(true_w0, color='k', ls='--', label='true')
ax[0].set_xlabel('image noise sigma')
ax[0].set_ylabel('$w_0$ estimate ($k_B T$)')
ax[0].set_title('noise increases parameter uncertainty')
ax[0].legend()
ax[1].plot(noise_levels, drifts, 'o-')
ax[1].set_xlabel('image noise sigma')
ax[1].set_ylabel('statistical distance')
ax[1].set_title('learned codebook drifts with noise')
plt.tight_layout()

## Exercises

1. **Noise and uncertainty.** Increase `noise_levels` to include 0.40 and 0.50. At what point does the fitted $w_0$ become biased, not just uncertain?
2. **Image area.** Change `n_atoms` from 48 to 24 and then to 72. Re-run the notebook sections that generate data and fit $w_0$. How does the block-wise spread change?
3. **Weaker segregation.** Set `true_w0 = 0.20`. Are the physics and k-means codebooks still able to distinguish the model from a random solid solution?
4. **Other loss functions** Change the loss function from the statistical distance to something like a KL divergence or even a mean squared loss. How does the interaction energy differ? (Use the physis codebook for this one). 
5. **Predictions for other T,compositions** The paper shows how you can use the model to predict what the microstates will be for other thermodyanmic conditions. See if you can do this as well. If you used a different loss function in (4), how far off is your prediction from the actual result, compared to if you use the statistical distance metric?